In [1]:
# --- CELL 1: Imports and Environment Setup ---
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score

# Set device to GPU if available, else CPU (or MPS for Mac)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")


Using device: cuda


In [2]:

# --- CELL 2: DataLoader Definition ---
class TUSZ_RNN(Dataset):
    """
    Modified TUSZ DataLoader for Simple RNNs.
    Outputs tensors in shape (Time, Channels) expected by PyTorch sequence models.
    """
    def __init__(self, path: str, mode: str, scale_factor: float = 1e4, processed_dir: str = "processed_tusz"):
        assert mode in ["train", "val", "test"], "Mode must be 'train', 'val', or 'test'"
        self.scale_factor = scale_factor
        self.data_path = os.path.join(path, processed_dir, mode)
        
        # Exception raising if path is missing
        if not os.path.isdir(self.data_path):
            raise FileNotFoundError(
                f"TUSZ {mode} folder not found at: {self.data_path}\n"
                f"Please verify your 'path' and 'processed_dir' variables."
            )

        x_files = sorted(f for f in os.listdir(self.data_path) if f.endswith("_X.npy"))
        if not x_files:
            raise FileNotFoundError(f"No *_X.npy files found in {self.data_path}. Did preprocessing complete successfully?")
        
        self.x_paths = [os.path.join(self.data_path, f) for f in x_files]

        # global index: sample i -> (recording file_ids[i], window win_ids[i])
        file_ids, win_ids, labels = [], [], []
        for fid, f in enumerate(x_files):
            y_path = os.path.join(self.data_path, f.replace("_X.npy", "_y.npy"))
            if not os.path.exists(y_path):
                raise FileNotFoundError(f"Missing label file: {y_path}")
            
            y = np.load(y_path)
            file_ids.append(np.full(len(y), fid, dtype=np.int32))
            win_ids.append(np.arange(len(y), dtype=np.int32))
            labels.append(y)
            
        self.file_ids = np.concatenate(file_ids)
        self.win_ids = np.concatenate(win_ids)
        self.labels = np.concatenate(labels).astype(np.int64)

    def __len__(self):
        return len(self.labels)

    def class_counts(self):
        return np.bincount(self.labels)

    def __getitem__(self, index):
        # Read from memmap. Original shape is (Channels=16, Time=1000)
        x = np.load(self.x_paths[int(self.file_ids[index])], mmap_mode="r")[int(self.win_ids[index])]
        
        # RNN expects (Time, Channels), so we transpose
        x = np.transpose(x, (1, 0)) 
        
        X = torch.from_numpy(np.array(x)).float() * self.scale_factor
        Y = torch.tensor(self.labels[index]).long()
        
        return X, Y


In [3]:

# # --- CELL 3: Model Architecture (Type A: Multivariate Simple RNN) ---
# class VanillaRNN(nn.Module):
#     def __init__(self, input_size=16, hidden_size=128, num_layers=2, num_classes=2):
#         super(VanillaRNN, self).__init__()
#         self.hidden_size = hidden_size
#         self.num_layers = num_layers
        
#         # Standard Elman RNN
#         # batch_first=True means input shape is (Batch, Time, Features)
#         self.rnn = nn.RNN(
#             input_size=input_size, 
#             hidden_size=hidden_size, 
#             num_layers=num_layers, 
#             nonlinearity='tanh', 
#             batch_first=True
#         )
        
#         # Fully connected layer to map final hidden state to classes
#         self.fc = nn.Linear(hidden_size, num_classes)

#     def forward(self, x):
#         # x shape: (Batch, Time=1000, Channels=16)
        
#         # out shape: (Batch, Time, Hidden_Size)
#         # hn shape: (Num_Layers, Batch, Hidden_Size)
#         out, _ = self.rnn(x)
        
#         # Extract the hidden state from the LAST time step (index -1)
#         last_time_step_out = out[:, -1, :] 
        
#         # Pass through the linear classifier
#         logits = self.fc(last_time_step_out)
#         return logits


In [4]:
# RNN with temporal averagepooling to deal with length of window due to vanishing gradients
class VanillaRNN(nn.Module):
    def __init__(self, input_size=16, hidden_size=128, num_layers=2, num_classes=2):
        super(VanillaRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # 1. Add Average Pooling to reduce sequence length from 1000 down to 100
        # Kernel size 10, stride 10 reduces T by a factor of 10.
        self.pool = nn.AvgPool1d(kernel_size=10, stride=10)
        
        self.rnn = nn.RNN(
            input_size=input_size, 
            hidden_size=hidden_size, 
            num_layers=num_layers, 
            nonlinearity='tanh', 
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x shape: (Batch, Time=1000, Channels=16)
        
        # AvgPool1d expects (Batch, Channels, Time), so we transpose back and forth
        x = x.transpose(1, 2)            # Now (Batch, 16, 1000)
        x = self.pool(x)                 # Now (Batch, 16, 100)
        x = x.transpose(1, 2)            # Now (Batch, 100, 16)
        
        out, _ = self.rnn(x)             # out shape: (Batch, 100, Hidden_Size)
        
        last_time_step_out = out[:, -1, :] 
        logits = self.fc(last_time_step_out)
        return logits

In [5]:
# class CNN_BiLSTM_Baseline(nn.Module):
#     def __init__(self, input_size=16, hidden_size=128, num_layers=2, num_classes=2):
#         super(CNN_BiLSTM_Baseline, self).__init__()
        
#         # 1D CNN Feature Extractor
#         # Instead of averaging 10 steps, we use a Conv layer with a kernel of 10 and stride of 5.
#         # This reduces the sequence length from 1000 down to 200, while LEARNING the features.
#         self.conv = nn.Conv1d(in_channels=input_size, out_channels=64, kernel_size=10, stride=5)
#         self.relu = nn.ReLU()
#         self.dropout = nn.Dropout(0.3) # Dropout prevents the larger model from overfitting
        
#         # BiLSTM Sequence Modeler
#         # Note: input_size is now 64 because the CNN outputs 64 feature maps
#         self.lstm = nn.LSTM(
#             input_size=64, 
#             hidden_size=hidden_size, 
#             num_layers=num_layers, 
#             batch_first=True,
#             bidirectional=True
#         )
        
#         self.fc = nn.Linear(hidden_size * 2, num_classes)

#     def forward(self, x):
#         # x shape: (Batch, Time=1000, Channels=16)
        
#         # CNN expects (Batch, Channels, Time)
#         x = x.transpose(1, 2)            # Now (Batch, 16, 1000)
        
#         # Pass through CNN feature extractor
#         x = self.conv(x)                 # Now (Batch, 64, 199)
#         x = self.relu(x)
#         x = self.dropout(x)
        
#         # LSTM expects (Batch, Time, Features)
#         x = x.transpose(1, 2)            # Now (Batch, 199, 64)
        
#         out, _ = self.lstm(x)            # out shape: (Batch, 199, 2 * Hidden_Size)
        
#         # Extract the hidden state from the LAST time step
#         last_time_step_out = out[:, -1, :] 
        
#         logits = self.fc(last_time_step_out)
#         return logits

In [6]:

# --- CELL 4: Training and Evaluation Functions ---
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item() * inputs.size(0)
            
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    epoch_loss = total_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    # Macro F1 is critical for TUSZ since background massively outweighs seizure windows
    epoch_f1 = f1_score(all_labels, all_preds, average='macro') 
    
    return epoch_loss, epoch_acc, epoch_f1

def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        # Gradient clipping prevents exploding gradients in simple RNNs
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) 
        optimizer.step()
        
        total_loss += loss.item() * inputs.size(0)
        
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
    epoch_loss = total_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')
    
    return epoch_loss, epoch_acc, epoch_f1


In [ ]:

# --- CELL 5: Main Execution Block ---
# Configuration and Hyperparameters
DATASET_ROOT = "/home/asifislam/Downloads/Datasets/TUHEEG/Seizure/v2.0.6/" # <-- UPDATE THIS TO YOUR ACTUAL ROOT PATH
PROCESSED_DIR = "processed_tusz" 

BATCH_SIZE = 64
LEARNING_RATE = 1e-4
NUM_EPOCHS = 30
HIDDEN_SIZE = 128
NUM_LAYERS = 2
INPUT_SIZE = 16 # 16 bipolar channels

# 1. Initialize Datasets
print("Loading datasets...")
train_dataset = TUSZ_RNN(path=DATASET_ROOT, mode="train", processed_dir=PROCESSED_DIR)
val_dataset = TUSZ_RNN(path=DATASET_ROOT, mode="val", processed_dir=PROCESSED_DIR)
test_dataset = TUSZ_RNN(path=DATASET_ROOT, mode="test", processed_dir=PROCESSED_DIR)

# Compute class weights for training 
class_counts = train_dataset.class_counts()
print(f"Train class counts (Background, Seizure): {class_counts}")
# Weight = Total Samples / (Number of Classes * Count per class)
class_weights = len(train_dataset) / (len(class_counts) * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

# 2. Initialize DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

# 3. Initialize Model, Loss, Optimizer
num_classes = len(class_counts)
model = VanillaRNN(input_size=INPUT_SIZE, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, num_classes=num_classes)
#model = CNN_BiLSTM_Baseline(input_size=INPUT_SIZE, hidden_size=HIDDEN_SIZE,num_layers=NUM_LAYERS,num_classes=num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 4. Training Loop
print(f"\nStarting training on {device}...")
best_val_f1 = 0.0

for epoch in range(NUM_EPOCHS):
    start_time = time.time()
    
    # Train
    train_loss, train_acc, train_f1 = train_epoch(model, train_loader, optimizer, criterion, device)
    
    # Validate
    val_loss, val_acc, val_f1 = evaluate(model, val_loader, criterion, device)
    
    # Test
    test_loss, test_acc, test_f1 = evaluate(model, test_loader, criterion, device)
    
    epoch_mins = (time.time() - start_time) / 60
    
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | Time: {epoch_mins:.2f}m")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Train Macro-F1: {train_f1:.4f}")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f} | Val Macro-F1:   {val_f1:.4f}")
    print(f"  Test Loss:  {test_loss:.4f} | Test Acc:  {test_acc:.4f} | Test Macro-F1:  {test_f1:.4f}")
    print("-" * 60)

print("Training Complete.")